# Compartment - gauss-only pipeline (Kaggle) - train80

Chay **train80** (compound-level 80/20 split) voi package gon `src/`: chi gauss head
(`N(mu, sigma^2)` per role), khong reg/softmax heads, khong MLM warmup.

### Setup / Quick start
1. **Repo**: private (AmnO-O/MoTune) -> set Kaggle Secret `GITHUB_TOKEN` (PAT co quyen doc repo).
   Repo public -> khong can. Muon chay fork rieng: sua `REPO_URL` / `REPO_BRANCH` o cell 1.
2. **Data**: tu dong doc tu Kaggle mount `/kaggle/input/datasets/ieltsmater/compartment/Compartment`.
3. **Weights (tu chon)**: set Kaggle Secret `HF_TOKEN` -> cell push model len HuggingFace
   (mac dinh `AmnO-O/compartment-weights`, co the doi `HF_REPO_ID` o cell 1).
4. **Run**: chay tuan tu TAT CA cac cell tu tren xuong.

### Data mix - bien `TRAIN_MIX` (cell 1)
- `'full'`  **(mac dinh)**: 4 lineages - en-nn + de-nn + en-pv + **de-pv** = **9825 rows / 1221 compound**.
  de-pv (trennbare Verben, mod=verb, head=particle): `_match_german_pv` tim duoc **100% spans**;
  ~965 rows (65%) non-degenerate (particle tach roi) co supervision that su; cac hang fused
  mot-token (`abgehauen`) tu dong bi mask `allowed`, chi lam representation.
- `'default'`: 3 lineages - en-nn + de-nn + en-pv = **8335 rows / 1063 compound**.
- `'nn'`:      chi en-nn + de-nn = **6778 rows / 905 compound** (tat het PV).

Cuoi cell train, log se in `Loaded NNNN total rows across NNNN unique compounds` -
so phai khop voi bang tren.

### Hyperparams
- `CONFIG_FILE` (env var, default ''): de `config/mae_gauss.json` khi muon dung config file
  (freeze 3 / lora 8 / ccc 0.7 / head_lr 2e-4 ...).
- `OVERRIDES` (cell 1): tune nhanh khong can sua file. Thu tu uu tien:
  default < config file < `OVERRIDES` < `TRAIN_EXTRA`.
- `TRAIN_EXTRA` (env var): `--set` bo sung, phan cach bang dau phay.

### Sau khi train (cac cell cuoi)
- Cell 4: in `metrics.json` (val_rho_mod / head / mean tren holdout 20%).
- Cell TRIAL: load `models/best.pt`, score cac file TRIAL that (en-nn-trial / en-pv-trial /
  de-nn-trial / de-pv-trial), ghi `submission/[lang]-[task]-pred.tsv` theo DUNG format submit
  (khong header, moi dong `ContextID <tab> score(s)`), zip thanh `submission.zip`.
  NN: 2 score (mod + head); PV: 1 score (mean cua 2 head).

In [ ]:
import os, subprocess, sys
from pathlib import Path

def _importable(name: str) -> bool:
    try:
        __import__(name)
        return True
    except Exception:
        return False

# ---- repo source -----------------------------------------------------------
REPO_URL = os.environ.get('REPO_URL', 'https://github.com/AmnO-O/MoTune.git')
REPO_BRANCH = os.environ.get('REPO_BRANCH', 'main')
GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN', '')    # set for private repos

dest = Path('/kaggle/working/Compartment')
url = REPO_URL
if GITHUB_TOKEN:
    url = url.replace('https://', f'https://{GITHUB_TOKEN}@')
dest.parent.mkdir(parents=True, exist_ok=True)
if (dest / 'src' / 'run.py').is_file():
    print('refreshing existing clone at', dest)
    subprocess.run(['git', '-C', str(dest), 'fetch', '--depth', '1', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(dest), 'reset', '--hard', f'origin/{REPO_BRANCH}'], check=True)
else:
    print('cloning', REPO_URL)
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, url, str(dest)], check=True)
REPO = dest
os.chdir(REPO)
print('repo:', REPO)
print('head:', subprocess.run(['git', '-C', REPO, 'log', '--oneline', '-1'],
                              capture_output=True, text=True).stdout.strip())

# ---- config file (optional) ------------------------------------------------
CONFIG_FILE = os.environ.get('CONFIG_FILE', '')   # e.g. 'config/mae_gauss.json'

# ---- knobs ----------------------------------------------------------------
MODE = os.environ.get('MODE', 'all')      # all | train80
if MODE not in ('all', 'train80'):
    MODE = 'train80'
TRAIN_EXTRA = os.environ.get('TRAIN_EXTRA', '')   # extra --set flags, comma-separated

# ---- tunable config overrides ('' = keep src.py default) ------------------
# Edit values here to tune a run without touching --set flags below.
OVERRIDES = {
    'freeze_epochs': '',
    'lora_epochs': '',
    'lora_rank': '',
    'lora_alpha': '',
    'lora_from_layer': '18',                # LoRA window: top layers (0 = all)
    'batch_size': '',
    'head_lr': '',
    'encoder_lr': '',
    'ccc_weight': '',
    'lambda_rank': '',
    'lambda_compound': '',
    'lambda_dist': '',                      # weight of KL(N(mu_p,sig_p)||N(y,sig_t))
    'bin_sigma': '',
    'num_workers': '',
    # --- pv lineage toggles (de '' o day; bat/tat bang TRAIN_MIX duoi) ---
    'en_pv_train': '',                       # non-empty -> FORCE it on regardless of TRAIN_MIX
    'de_pv_train': '',                       # 'de-pv-train.tsv' adds German PV (~65% supervised; fused one-token rows masked)
    'aux_data_paths': '',                     # e.g. 'nctti_en.tsv' label-free rows
    # --- gauss feature / context knobs ---
    'context_layers': '',                     # e.g. 10,16,22 = mean-pool 3 global layers ('' = last)
    'span_layers': '',                        # e.g. -1 = last layer ('' = auto mid-5)
    'gauss_dedicated': '',                     # '1' = mod/head gauss heads read context at their OWN layer
    'gauss_ctx_mod': '',                        # context layer for modifier head (e.g. 19 = block 18 global)
    'gauss_ctx_head': '',                       # context layer for head-noun head (e.g. 20 = block 19 local)
    'gauss_ctx_pv': '',                         # en-pv rows -> deepest global layer (e.g. 22)
}

def cfg_sets(*extra):
    out = []
    for k, v in OVERRIDES.items():
        if v not in (None, ''):
            out += ['--set', f'{k}={v}']
    for e in extra:
        out += ['--set', e]
    return out

# ---- data mix: lineages tham gia train80 -----------------------------------
# 'full'    = 4 lineages: en-nn + de-nn + en-pv + de-pv
#            (9825 rows / 1221 c; de-pv trennbare ~965 supervised)
# 'default' = en-nn + de-nn + en-pv                    (8335 rows / 1063 c)
# 'nn'      = en-nn + de-nn only                       (6778 rows / 905 c)
# Cuoi cell train, log in 'Loaded NNNN total rows ...' de xac nhan mix.
TRAIN_MIX = os.environ.get('TRAIN_MIX', 'full')
EXTRA_SETS: list = []
if TRAIN_MIX == 'nn' and not OVERRIDES.get('en_pv_train'):
    EXTRA_SETS += ['en_pv_train=', 'de_pv_train=']   # 'key=' -> empty string = lineage off
if TRAIN_MIX in ('nn', 'default') and not OVERRIDES.get('de_pv_train') and 'de_pv_train=' not in EXTRA_SETS:
    EXTRA_SETS.append('de_pv_train=')                # drop German PV
# 'full' can them gi: de_pv_train='de-pv-train.tsv' la config default cua src/.
# OVERRIDES non-empty ('en_pv_train'/'de_pv_train') thang TRAIN_MIX.

# ---- memory ---------------------------------------------------------------
# expandable segments reduce T4 fragmentation; inherited by the subprocess
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')

# ---- HuggingFace push (optional) -----------------------------------------
HF_REPO_ID = os.environ.get('HF_REPO_ID', 'AmnO-O/compartment-weights')
HF_PRIVATE = os.environ.get('HF_PRIVATE', 'true').lower() not in ('0','false','no')

def _hf_token() -> str:
    tok = os.environ.get('HF_TOKEN', '')
    if tok:
        return tok
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        return ''

# ---- ensure imports exist (Kaggle already ships them) ---------------------
for m in ('torch', 'transformers', 'pandas', 'numpy', 'sklearn', 'scipy', 'yaml'):
    if not _importable(m):
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', m], check=False)

def run(args):
    print('\n>>>', ' '.join(args))
    subprocess.run(args, cwd=REPO, check=True)


In [ ]:
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# --- train80: single compound-level 80/20 split ---------------------------
# gauss-only package: python -m src.run  (no warmup / no reg or softmax heads)
if MODE in ('all', 'train80'):
    args = [
        sys.executable, '-m', 'src.run', '--device', 'cuda',
        '--set', 'num_workers=0',            # chống deadlock Kaggle
        '--set', 'batch_size=64',
        '--set', 'freeze_epochs=3',
        '--set', 'head_lr=2e-4',
    ]

    if CONFIG_FILE:
        args += ['--config', CONFIG_FILE]

    args += list(cfg_sets(*EXTRA_SETS))   # OVERRIDES then TRAIN_MIX over the config file

    if TRAIN_EXTRA:
        args += [x.strip() for x in TRAIN_EXTRA.split(',') if x.strip()]

    mix_rows = {'full': '9825 / 1221 compounds', 'default': '8335 / 1063 compounds',
                'nn': '6778 / 905 compounds'}
    print('>>> MODE=%s TRAIN_MIX=%s (%s) CONFIG_FILE=%s' % (
        MODE, TRAIN_MIX, mix_rows.get(TRAIN_MIX, '?'), CONFIG_FILE or '(defaults)'))
    run(args)


In [ ]:
# --- Push trained weights to HuggingFace --------------------------------
# Needs only HF_TOKEN (Kaggle Secret or env var) + HF_REPO_ID set above.
# Skipped automatically when HF_REPO_ID is empty.
if HF_REPO_ID and MODE in ('all', 'train80'):
    if not _hf_token():
        print('SKIP push: no HF_TOKEN found (set Kaggle Secret named HF_TOKEN).')
    else:
        from huggingface_hub import HfApi
        api = HfApi(token=_hf_token())
        api.create_repo(repo_id=HF_REPO_ID, private=HF_PRIVATE, exist_ok=True, repo_type='model')
        work = Path('/kaggle/working')
        # upload all model checkpoints
        models_dir = work / 'models'
        if models_dir.is_dir():
            api.upload_folder(
                folder_path=str(models_dir),
                repo_id=HF_REPO_ID,
                path_in_repo='models',
                allow_patterns='*.pt'
            )
            pt_files = list(models_dir.glob('*.pt'))
            print(f'pushed {len(pt_files)} model file(s)')
        # upload key artifacts
        for name in ('config.json', 'metrics.json', 'history.json'):
            src = work / name
            if src.is_file():
                api.upload_file(path_or_fileobj=str(src), path_in_repo=name, repo_id=HF_REPO_ID)
                print(f'pushed {name}')
        print(f'HF repo: https://huggingface.co/{HF_REPO_ID}')
else:
    print('push skipped (HF_REPO_ID empty or MODE=%s)' % MODE)


In [ ]:
import json

work = Path('/kaggle/working')
p = work / 'metrics.json'
if p.is_file():
    print('--- metrics.json (val rho on holdout 20%) ---')
    print(json.dumps(json.loads(p.read_text(encoding='utf-8')), indent=2))
else:
    print('no metrics.json yet')


In [ ]:
# --- TRIAL after training: reload best.pt, score the REAL per-lineage trial files ----
# TRAI files la holdout rieng tung lineage (en-nn-trial / en-pv-trial / de-nn-trial /
# de-pv-trial). KHONG re-split train de lam trial - each lineage has its own file.
# Ghi submission theo DUNG format: zip gom cac file [language]-[task]-pred.tsv,
# KHONG header, moi dong: ContextID <tab> score(s) (NN: mod + head; PV: 1 score = mean
# cua 2 head). Moi lineage mot file pred rieng. Tu dong BO QUA neu nochua best.pt.
import sys, json, logging, torch, zipfile
from pathlib import Path

work = Path('/kaggle/working')
ckpt = work / 'models' / 'best.pt'
if not ckpt.is_file():
    for cand in (REPO / 'models' / 'best.pt', REPO / 'output' / 'models' / 'best.pt', Path('output/models/best.pt'), Path('models/best.pt')):
        if cand.is_file():
            ckpt = cand
            work = cand.parent.parent
            break
if not ckpt.is_file():
    print('SKIP trial: chua co models/best.pt (train chua xong).')
else:
    sys.path.insert(0, str(REPO))
    from src.config import Config
    from src.pipeline import _tokenizer
    from src.data import load_trial, CompDataset, collate_comp
    from src.model import apply_lora, build_model
    from src.train import evaluate
    from torch.utils.data import DataLoader
    from scipy.stats import spearmanr

    lg = logging.getLogger('trial')
    cfg_p = work / 'config.json'
    cfg = (Config() if not cfg_p.is_file()
           else Config(**json.loads(cfg_p.read_text(encoding='utf-8'))))
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tok = _tokenizer(cfg, lg)

    # best.pt duoc luu KHI LoRA adapters dang active (trainer._apply_lora wrap
    # top layers TRUOC khi train) -> phai wrap lai nhu the TRUOC load_state_dict,
    # nguoc lai moi lora key se mismatch (RuntimeError missing/unexpected).
    model = build_model(cfg, device)
    apply_lora(model, rank=cfg.lora_rank, alpha=cfg.lora_alpha,
               dropout=cfg.lora_dropout, targets=cfg.lora_targets,
               from_layer=cfg.lora_from_layer)
    model.load_state_dict(torch.load(ckpt, map_location=device))
    model.eval()

    lineage = load_trial(cfg)
    sub = work / 'submission'
    sub.mkdir(parents=True, exist_ok=True)
    for key, rows in lineage.items():
        ds = CompDataset(rows, tok, max_len=cfg.max_context_length)
        loader = DataLoader(ds, batch_size=64, shuffle=False, collate_fn=collate_comp)
        # return_all=True -> (mod, head, mod_y, head_y, mask), row-aligned voi rows.
        mp, hp, my, hy, mask = evaluate(model, loader, device, return_all=True)
        assert len(mp) == len(rows), (key, len(mp), len(rows))
        pv = bool(rows[0]['is_pv'])
        score = 0.5 * (mp + hp) if pv else mp   # PV: 1 score = mean 2 head
        lines = []
        for r, pm, ph, sc, gm, gh in zip(rows, mp, hp, score, my, hy):
            if pv:
                lines.append('%s\t%.4f' % (r['context_id'], float(sc)))
            else:
                lines.append('%s\t%.4f\t%.4f' % (r['context_id'], float(pm), float(ph)))
        fname = '%s-pred.tsv' % key          # vi du: en-nn-pred.tsv / de-pv-pred.tsv
        (sub / fname).write_text('\n'.join(lines), encoding='utf-8')
        msk = mask.astype(bool)
        if int(msk.sum()) > 0:
            if pv:
                rho = spearmanr(my[msk], score[msk]).correlation
                print('%s: %d rows | rho=%.4f' % (key, len(rows), rho))
            else:
                rho_m = spearmanr(my[msk], mp[msk]).correlation
                rho_h = spearmanr(hy[msk], hp[msk]).correlation
                print('%s: %d rows | rho mod=%.4f head=%.4f avg=%.4f'
                      % (key, len(rows), rho_m, rho_h, (rho_m + rho_h) / 2.0))
        else:
            print('%s: %d rows | khong co label (chi ghi prediction)' % (key, len(rows)))

    zip_p = work / 'submission.zip'          # archive dung format: cac *-pred.tsv o root
    with zipfile.ZipFile(zip_p, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in sorted(sub.glob('*-pred.tsv')):
            zf.write(f, arcname=f.name)
    print('submission files ->', sorted(f.name for f in sub.glob('*-pred.tsv')))
    print('zipped ->', zip_p)

### Lưu ý
- Tín hiệu tốt: `val_rho_mean` (rho trung bình Mod/Head trên holdout 20%, split theo compound).
- `gauss_dedicated=1` + `gauss_ctx_mod=19` + `gauss_ctx_head=20` + `gauss_ctx_pv=22` = variant đang test (config/mae_gauss.json).
- Data mix: `TRAIN_MIX` = full (9825 rows, de-pv ~97% representation-only) / default (8335) / nn (6778).
- predict / train5 / resume chưa wire trong `src/` (chạy qua `mm/` cũ nếu cần). CELL TRIAL chỉ tái score holdout để xem prediction.